# 必要パッケージのインポート

In [ ]:
import requests
# from elasticsearch8 import Elasticsearch
from sentence_transformers import SentenceTransformer
import json
import time

# 設定

In [ ]:
ES_URL = 'http://llm-rag-examples-elasticsearch1:9200'
INDEX_NAME = 'vector_test01'
HEADERS = {
    'Accept': 'application/vnd.elasticsearch+json; compatible-with=8',
    'Content-Type': 'application/vnd.elasticsearch+json; compatible-with=8',
}

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
MODEL_DIM = 384

# 初期化

In [ ]:
# 初期化
model = SentenceTransformer(MODEL_NAME)

# インデックス作成

In [ ]:
res = requests.head(f"{ES_URL}/{INDEX_NAME}")
exists_index = res.status_code == 200
exists_index

In [ ]:
if exists_index:
    print(f"Index '{INDEX_NAME}' already exists.")
else:
    mapping = {
        'mappings': {
            'properties': {
                'text': {'type': 'text'},
                'vector': {
                    'type': 'dense_vector',
                    'dims': MODEL_DIM,
                    'index': True,
                    'similarity': 'cosine'
                }
            }
        }
    }
    res = requests.put(f"{ES_URL}/{INDEX_NAME}", headers=HEADERS, data=json.dumps(mapping))
    res.raise_for_status()
    print(f"Index '{INDEX_NAME}' created.")

# ドキュメントをインデックス

In [ ]:
# テキスト登録
def index_text(text):
    vector = model.encode(text)
    doc = {
        "text": text,
        "vector": vector.tolist()
    }
    res = requests.post(f"{ES_URL}/{INDEX_NAME}/_doc", headers=HEADERS, data=json.dumps(doc))
    res.raise_for_status()
    print(f"Indexed: {text}")

In [ ]:
# --- 登録するテキストデータ ---
texts = [
    '猫は可愛い動物です。',
    '犬は人間の親友と呼ばれています。',
    '東京は日本の首都です。'
]

for text in texts:
    index_text(text)

# ベクトル検索

In [ ]:
# 類似検索
def search_similar_text(query_text, top_k=3):
    query_vector = model.encode(query_text)
    query = {
        'knn': {
            'field': 'vector',
            'query_vector': query_vector.tolist(),
            'k': top_k,
            'num_candidates': 100
        }
    }
    body = {
        'size': top_k,
        'query': query
    }
    res = requests.post(f"{ES_URL}/{INDEX_NAME}/_search", headers=HEADERS, data=json.dumps(body))
    res.raise_for_status()

    hits = res.json()['hits']['hits']
    print("--- 検索結果 ---")
    for hit in hits:
        print(f"スコア: {hit['_score']:.4f}, テキスト: {hit['_source']['text']}")

In [ ]:
time.sleep(5)

In [ ]:
QUERY_TEXT = '日本の都市'

In [ ]:
search_similar_text(QUERY_TEXT)